# Recherche b5 — six hypothèses autour de weights-b4

**Notebook de recherche séparé — il ne remplace jamais la soumission.**

Ce notebook est prévu pour Google Colab CPU. Il entraîne six configurations nouvelles à partir du
split `train` officiel uniquement, puis les évalue sur les 24 000 lignes `validation` avec :

1. le package `competition` du dépôt, utilisé par `competition.evaluation` et le leaderboard ;
2. le helper officiel `starter/utils.py` épinglé au commit `75578f2400c39b1f8e31ce7e7104b37fbc470d11` ;
3. un contrôle de reconstruction strict, sans NFC ni suppression d’espaces ;
4. la règle de temps du leaderboard lorsqu’une baseline locale est disponible.

Les six expériences sont :

| Candidat | en | fr | ha | sw | yo | am | min_frequency |
|---|---:|---:|---:|---:|---:|---:|---:|
| `fr1125-yo5` | 1 | 1,125 | 4 | 4 | 5 | 4 | 5 |
| `fr125-yo5` | 1 | 1,25 | 4 | 4 | 5 | 4 | 5 |
| `fr1125-yo5-am5` | 1 | 1,125 | 4 | 4 | 5 | 5 | 5 |
| `fr125-yo5-am5` | 1 | 1,25 | 4 | 4 | 5 | 5 | 5 |
| `b4-minfreq3` | 1 | 1 | 4 | 4 | 4 | 4 | 3 |
| `b4-minfreq7` | 1 | 1 | 4 | 4 | 4 | 4 | 7 |

Les poids fractionnaires utilisent la méthode cumulative de l’ancien notebook, généralisée en huitièmes :
`8 = ×1`, `9 = ×1,125`, `10 = ×1,25`. Les copies supplémentaires sont distribuées par index de ligne
au sein de chaque langue, jamais selon le contenu ni selon la validation.

Le tokenizer b4 soumis est utilisé comme **référence déjà produite** : il est téléchargé depuis un commit
immuable et réévalué, mais il n’est pas réentraîné dans cette grille. Les artefacts sont écrits hors de
`submissions/`. Aucun fichier de soumission, README ou metadata n’est remplacé automatiquement.

## Politique d’évaluation

Le score principal est celui de `competition.metrics.score_tokenizer`, appelé par
`competition.evaluation.evaluate_submission`. Le rapport du helper épinglé est conservé séparément et
son score doit être cohérent avec le score principal.

Le notebook ne remplace pas le score par une politique locale :

- le score complet et ses pénalités restent visibles ;
- la marge EN/FR est calculée à partir des valeurs exactes du b4 réévalué ;
- une configuration n’est dite admissible que si son score est strictement plus bas et sa marge au moins
  égale à celle du b4, avec zéro UNK, zéro pénalité et reconstruction stricte complète ;
- les valeurs historiques `1.939666974` et `1.787199439 %` ne sont jamais injectées dans le calcul ;
- la validation a déjà servi à sélectionner des recettes : les résultats restent des mesures publiques,
  pas une garantie de score privé ou de classement.

Les fixtures du dépôt ne sont pas utilisées pour les résultats. Si le dataset Hugging Face ou le commit du
checker ne sont pas accessibles, le notebook s’arrête au lieu de les remplacer discrètement.

In [ ]:
# Installation et chargement du code d’évaluation du dépôt.
import base64
import csv
import gc
import hashlib
import importlib.util
from importlib import metadata as importlib_metadata
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import time
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

INSTALL = [
    "tokenizers==0.22.1",
    "datasets>=4,<5",
    "PyYAML==6.0.2",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *INSTALL], check=True)

TOKENIZERS_VERSION = "0.22.1"
if importlib.util.find_spec("tokenizers") is None:
    raise RuntimeError("tokenizers n’est pas disponible après installation")
import tokenizers
if tokenizers.__version__ != TOKENIZERS_VERSION:
    raise RuntimeError(f"tokenizers=={TOKENIZERS_VERSION} requis, trouvé {tokenizers.__version__}")

def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "competition").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    clone = Path("/content/amtc-official-evaluator")
    if not (clone / "competition").is_dir():
        if clone.exists():
            shutil.rmtree(clone)
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge.git",
            str(clone),
        ], check=True)
    return clone

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))
try:
    EVALUATOR_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
    ).strip()
except Exception:
    EVALUATOR_COMMIT = "unavailable"
import competition
from competition.constants import (
    CONTEXT_FERTILITY_RATIO,
    LANGUAGES,
    MAX_TOKENIZER_BYTES,
    RECONSTRUCTION_PENALTY,
    SCORED_LANGUAGES,
    SUPPORTED_TOKENIZERS_VERSION,
)
from competition.data import Example
from competition.evaluation import evaluate_submission
from competition.hub_data import load_public_split
from competition.leaderboard import build_leaderboard
from competition.validation import validate_tokenizer

print("tokenizers:", tokenizers.__version__)
print("competition:", competition.__version__)
print("évaluateur:", REPO_ROOT)


In [ ]:
# Configuration figée avant l’exécution.
USE_GOOGLE_DRIVE = False  # True uniquement si l’on veut conserver les modèles entre sessions Colab.
RUN_LABEL = "b5-weight-grid-v1"  # Ne pas modifier pour reprendre exactement cette recherche.
CPU_THREADS = 2
DENOMINATOR = 8

DATASET_ID = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"
CHECKER_COMMIT = "75578f2400c39b1f8e31ce7e7104b37fbc470d11"
CHECKER_SHA256 = "1727de34136097eb48addabf90501589bdfefa31c20e201bab53c38f2f7c9688"
REFERENCE_COMMIT = "0d9ba4bccbc5c5a5fc48ed76d655ba4211ee2749"
REFERENCE_SHA256 = "b2f9a461ce1b0e8bff07c00d982614f49d555464b8fb1efa46e66b08b67013e4"
REFERENCE_URL = (
    "https://raw.githubusercontent.com/maick-code/airf-multilingual-tokenizer-challenge/"
    f"{REFERENCE_COMMIT}/submissions/maick-dane-nkou/tokenizer.json"
)
CHECKER_URL = (
    "https://raw.githubusercontent.com/aims-ai-research-foundations/"
    f"airf-multilingual-tokenizer-challenge/{CHECKER_COMMIT}/starter/utils.py"
)

# Les unités sont dans l’ordre en, fr, ha, sw, yo, am.
def config(name, units, min_frequency):
    if len(units) != len(LANGUAGES):
        raise ValueError(name)
    if any(type(value) is not int or value < DENOMINATOR for value in units):
        raise ValueError(f"Poids invalides pour {name}")
    return {
        "name": name,
        "units": dict(zip(LANGUAGES, units, strict=True)),
        "denominator": DENOMINATOR,
        "vocab_size": 10_000,
        "min_frequency": min_frequency,
        "architecture": "space_word_v1",
    }

CONFIGS = [
    config("fr1125-yo5",       (8, 9, 32, 32, 40, 32), 5),
    config("fr125-yo5",         (8, 10, 32, 32, 40, 32), 5),
    config("fr1125-yo5-am5",    (8, 9, 32, 32, 40, 40), 5),
    config("fr125-yo5-am5",     (8, 10, 32, 32, 40, 40), 5),
    config("b4-minfreq3",       (8, 8, 32, 32, 32, 32), 3),
    config("b4-minfreq7",       (8, 8, 32, 32, 32, 32), 7),
]
if len({item["name"] for item in CONFIGS}) != 6:
    raise ValueError("La grille doit contenir exactement six noms distincts")

BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
WORK_DIR = BASE / "airf-tokenizer-b5-runtime"
SAVE_ROOT = BASE / "airf-tokenizer-b5-results"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_ROOT = Path("/content/drive/MyDrive/airf-tokenizer/b5-weight-grid")
WORK_DIR.mkdir(parents=True, exist_ok=True)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
for path in (WORK_DIR, SAVE_ROOT):
    if "submissions" in path.resolve().parts:
        raise ValueError("Les sorties ne doivent jamais être dans submissions/")

os.environ["RAYON_NUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"
print("Six configurations :", [item["name"] for item in CONFIGS])
print("Calcul/cache :", WORK_DIR)
print("Résultats :", SAVE_ROOT)


## Chargement des données officielles

Le train est sérialisé en JSONL dans l’ordre round-robin `en/fr/ha/sw/yo/am`, en conservant l’ordre
interne de chaque langue. Les répétitions sont générées à la volée par le travailleur CPU.

La validation est sérialisée séparément et ne reçoit jamais le chemin du train. Les deux splits sont
contrôlés à 40 000 et 4 000 lignes par langue. Le notebook conserve les empreintes Hugging Face et les
SHA-256 des sérialisations utilisées pour la reprise.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)

def dump_pair(stream, language, text):
    stream.write(json.dumps([language, text], ensure_ascii=False, separators=(",", ":")) + "\n")

def prepare_split(split, destination):
    if split not in {"train", "validation"}:
        raise ValueError(split)
    dataset = load_public_split(split, revision=DATASET_REVISION, streaming=False)
    expected_per_language = 40_000 if split == "train" else 4_000
    counts = Counter()
    fingerprint = getattr(dataset, "_fingerprint", None)
    destination = Path(destination)
    temporary = destination.with_name(destination.name + ".partial")
    with tempfile.TemporaryDirectory(dir=WORK_DIR) as temporary_dir:
        temporary_dir = Path(temporary_dir)
        language_paths = {lang: temporary_dir / f"{lang}.jsonl" for lang in LANGUAGES}
        streams = {}
        try:
            if split == "train":
                streams = {lang: path.open("w", encoding="utf-8", newline="\n")
                           for lang, path in language_paths.items()}
            with temporary.open("w", encoding="utf-8", newline="\n") as output:
                for row in dataset:
                    language = row.get("language")
                    text = row.get("text")
                    if language not in LANGUAGES or not isinstance(text, str) or not text.split():
                        raise ValueError(f"Ligne invalide dans {split}")
                    counts[language] += 1
                    dump_pair(streams[language] if split == "train" else output, language, text)
                expected = dict.fromkeys(LANGUAGES, expected_per_language)
                if counts != expected:
                    raise ValueError(f"Comptes inattendus pour {split}: {dict(counts)}")
                if split == "train":
                    for stream in streams.values():
                        stream.close()
                    readers = [path.open(encoding="utf-8") for path in language_paths.values()]
                    try:
                        for group in zip(*readers, strict=True):
                            output.writelines(group)
                    finally:
                        for reader in readers:
                            reader.close()
        finally:
            for stream in streams.values():
                stream.close()
    os.replace(temporary, destination)
    result = {
        "split": split,
        "rows": sum(counts.values()),
        "counts": dict(counts),
        "fingerprint": str(fingerprint),
        "sha256": sha256_file(destination),
        "serialization": "jsonl-language-text-v1; train round-robin" if split == "train"
                         else "jsonl-language-text-v1; validation source order",
    }
    del dataset
    gc.collect()
    print(split, result)
    return result

TRAIN_PATH = WORK_DIR / "train.jsonl"
VALIDATION_PATH = WORK_DIR / "validation.jsonl"
VALIDATION_CSV = WORK_DIR / "validation.csv"
TRAIN_INFO = prepare_split("train", TRAIN_PATH)
VALIDATION_INFO = prepare_split("validation", VALIDATION_PATH)

def jsonl_to_csv(source, destination):
    temporary = Path(destination).with_name(Path(destination).name + ".partial")
    with Path(source).open(encoding="utf-8") as source_stream, temporary.open(
        "w", encoding="utf-8", newline=""
    ) as destination_stream:
        writer = csv.writer(destination_stream)
        writer.writerow(["language", "text"])
        for line in source_stream:
            language, text = json.loads(line)
            writer.writerow([language, text])
    os.replace(temporary, destination)

jsonl_to_csv(VALIDATION_PATH, VALIDATION_CSV)


In [ ]:
# Contexte immuable de cette recherche.
CONTEXT = {
    "dataset": DATASET_ID,
    "revision": DATASET_REVISION,
    "train": TRAIN_INFO,
    "validation": VALIDATION_INFO,
    "tokenizers_version": tokenizers.__version__,
    "datasets_version": importlib_metadata.version("datasets"),
    "python_version": sys.version.split()[0],
    "cpu_threads": CPU_THREADS,
    "checker_commit": CHECKER_COMMIT,
    "checker_sha256": CHECKER_SHA256,
    "configs": CONFIGS,
    "worker_denominator": DENOMINATOR,
    "evaluator_commit": EVALUATOR_COMMIT,
}
context_id = hashlib.sha256(json.dumps(CONTEXT, sort_keys=True).encode()).hexdigest()[:16]
SESSION_DIR = SAVE_ROOT / f"{RUN_LABEL}-{context_id}"
SESSION_DIR.mkdir(parents=True, exist_ok=True)
atomic_write_text(SESSION_DIR / "run_context.json", json.dumps(CONTEXT, ensure_ascii=False, indent=2) + "\n")
print("Dossier de session :", SESSION_DIR)


## Checker épinglé et référence b4

Le helper est téléchargé à partir du commit immuable et son SHA-256 est vérifié avant toute évaluation.
La référence b4 est également téléchargée depuis le commit de la PR #11 et son SHA est contrôlé.
Une divergence arrête le notebook : aucun fichier de remplacement n’est utilisé.

In [ ]:
from importlib import metadata as importlib_metadata

# Le contexte a déjà été écrit avec la version datasets dans la cellule précédente.
# Cette vérification explicite est conservée dans le rapport final.
if importlib_metadata.version("datasets") != CONTEXT["datasets_version"]:
    raise RuntimeError("Version datasets modifiée pendant la session")

def fetch_bytes(url):
    with urllib.request.urlopen(url, timeout=120) as response:
        return response.read()

checker_bytes = fetch_bytes(CHECKER_URL)
if hashlib.sha256(checker_bytes).hexdigest() != CHECKER_SHA256:
    raise RuntimeError("SHA-256 du checker officiel incorrect")
CHECKER_PATH = WORK_DIR / "official_utils.py"
CHECKER_PATH.write_bytes(checker_bytes)
checker_spec = importlib.util.spec_from_file_location("pinned_official_utils", CHECKER_PATH)
official_utils = importlib.util.module_from_spec(checker_spec)
checker_spec.loader.exec_module(official_utils)
if official_utils.REQUIRED_TOKENIZERS_VERSION != TOKENIZERS_VERSION:
    raise RuntimeError("Contrat tokenizers inattendu dans le checker")

REFERENCE_DIR = SESSION_DIR / "reference"
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_PATH = REFERENCE_DIR / "tokenizer.json"
if REFERENCE_PATH.exists():
    if sha256_file(REFERENCE_PATH) != REFERENCE_SHA256:
        raise RuntimeError("Référence b4 sauvegardée avec un SHA inattendu")
else:
    REFERENCE_PATH.write_bytes(fetch_bytes(REFERENCE_URL))
if sha256_file(REFERENCE_PATH) != REFERENCE_SHA256:
    raise RuntimeError("Le tokenizer b4 téléchargé ne correspond pas au SHA attendu")

# Le contrôle local ne crée jamais de fichier dans submissions/.
EVALUATION_SUBMISSIONS = SESSION_DIR / "evaluation_inputs"
EVALUATION_SUBMISSIONS.mkdir(parents=True, exist_ok=True)
print("Checker vérifié :", CHECKER_COMMIT, CHECKER_SHA256)
print("Référence b4 vérifiée :", REFERENCE_SHA256)


## Travailleur CPU isolé

Chaque entraînement est exécuté dans un processus séparé. Le travailleur ne reçoit que `train.jsonl`,
la configuration et un chemin de sortie. Il ne connaît pas `validation.jsonl`.

Les poids utilisent la formule cumulative de l’ancien notebook : pour une langue et une ligne d’index `i`,
le nombre de copies est

```text
floor((i + 1) × unités / dénominateur) - floor(i × unités / dénominateur)
```

Avec un dénominateur de 8, chaque ligne originale apparaît au moins une fois et les fractions sont
exactes sur 40 000 lignes par langue.

In [ ]:
WORKER_SOURCE = r"""import argparse
import hashlib
import json
import os
import time
from collections import Counter
from pathlib import Path

from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers, trainers

LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
DENOMINATOR = 8
TOKENIZERS_VERSION = "0.22.1"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def read_rows(path):
    with Path(path).open(encoding="utf-8") as stream:
        for line in stream:
            language, text = json.loads(line)
            if language not in LANGUAGES or not isinstance(text, str) or not text.split():
                raise ValueError("Invalid train row")
            yield language, text


def weighted_texts(rows, units):
    if set(units) != set(LANGUAGES):
        raise ValueError("Missing language weight")
    if any(type(value) is not int or value < DENOMINATOR for value in units.values()):
        raise ValueError("Invalid cumulative weight")
    seen = Counter()
    for language, text in rows:
        index = seen[language]
        seen[language] += 1
        copies = ((index + 1) * units[language]) // DENOMINATOR - (index * units[language]) // DENOMINATOR
        if copies < 1:
            raise ValueError("A train row was omitted")
        for _ in range(copies):
            yield text


def train_tokenizer(texts, *, vocab_size, min_frequency, length):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Split(Regex(r" ?\S+|\s+"), behavior="isolated"),
        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
    ])
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        special_tokens=[],
        initial_alphabet=sorted(pre_tokenizers.ByteLevel.alphabet()),
        show_progress=True,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer, length=length)
    return tokenizer


ROUNDTRIP_CASES = [
    "", " ", "   ", "\t\n\r\n", "  Hello  WORLD!\tNext\nline.  ",
    "[UNK] [CLS] [SEP] <0xFF>", "é e\u0301 Ì I\u0300", "👩🏿‍💻 🌍 中文 العربية",
    "a\u00a0b\u2003c\u200bd", "\x00\x01\x7f\ufeff\U0010ffff", "don't l’amour — … ።",
]


def strict_failure_count(tokenizer, texts, batch_size=512):
    failures = 0
    texts = list(texts)
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encodings = tokenizer.encode_batch(batch, add_special_tokens=False)
        ids = [encoding.ids for encoding in encodings]
        decoded = tokenizer.decode_batch(ids, skip_special_tokens=False)
        skipped = tokenizer.decode_batch(ids, skip_special_tokens=True)
        failures += sum(
            original != restored or original != restored_skipped
            for original, restored, restored_skipped in zip(batch, decoded, skipped, strict=True)
        )
    return failures


def train_job(train_path, config_path, output):
    config = json.loads(Path(config_path).read_text(encoding="utf-8"))
    if config["vocab_size"] != 10_000 or config["architecture"] != "space_word_v1":
        raise ValueError("Unexpected architecture configuration")
    if config["denominator"] != DENOMINATOR:
        raise ValueError("Unexpected weight denominator")
    counts = Counter(language for language, _ in read_rows(train_path))
    if counts != dict.fromkeys(LANGUAGES, 40_000):
        raise ValueError(f"Full official train required: {dict(counts)}")
    weighted_rows = sum(40_000 * config["units"][language] // DENOMINATOR for language in LANGUAGES)
    before = sha256_file(train_path)
    started = time.perf_counter()
    tokenizer = train_tokenizer(
        weighted_texts(read_rows(train_path), config["units"]),
        vocab_size=config["vocab_size"],
        min_frequency=config["min_frequency"],
        length=weighted_rows,
    )
    if tokenizer.get_vocab_size(with_added_tokens=True) != config["vocab_size"]:
        raise ValueError("Expected exactly 10,000 vocabulary entries")
    temporary = Path(output).with_name(Path(output).name + ".partial")
    tokenizer.save(str(temporary), pretty=True)
    del tokenizer
    reloaded = Tokenizer.from_file(str(temporary))
    if strict_failure_count(reloaded, ROUNDTRIP_CASES):
        raise ValueError("Lossy pipeline on strict regression cases")
    if Path(temporary).stat().st_size > 20 * 1024 * 1024:
        raise ValueError("Tokenizer exceeds 20 MiB")
    if sha256_file(train_path) != before:
        raise ValueError("Train corpus changed during training")
    os.replace(temporary, output)
    atomic_json(Path(output).parent / "training.json", {
        "config": config,
        "train_sha256": before,
        "tokenizer_sha256": sha256_file(output),
        "training_seconds": time.perf_counter() - started,
        "weighted_rows": weighted_rows,
    })


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("train_path")
    parser.add_argument("config_path")
    parser.add_argument("output")
    args = parser.parse_args()
    from tokenizers import __version__
    if __version__ != TOKENIZERS_VERSION:
        raise RuntimeError("tokenizers==0.22.1 required")
    train_job(args.train_path, args.config_path, args.output)


if __name__ == "__main__":
    main()
"""
WORKER_PATH = WORK_DIR / "cpu_train_worker.py"
WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
WORKER_SHA256 = sha256_file(WORKER_PATH)
print("Travailleur SHA-256 :", WORKER_SHA256)


In [ ]:
# Régression jetable sur les exemples de contrôle seulement.
worker_spec = importlib.util.spec_from_file_location("b5_cpu_worker", WORKER_PATH)
worker = importlib.util.module_from_spec(worker_spec)
worker_spec.loader.exec_module(worker)
mini = worker.train_tokenizer(
    worker.ROUNDTRIP_CASES,
    vocab_size=512,
    min_frequency=1,
    length=len(worker.ROUNDTRIP_CASES),
)
if worker.strict_failure_count(mini, worker.ROUNDTRIP_CASES) != 0:
    raise RuntimeError("La régression réversible a échoué")
del mini
print("Régression stricte réussie :", len(worker.ROUNDTRIP_CASES), "cas")


## Évaluation complète d’un tokenizer

Pour chaque modèle, le notebook :

1. le place dans un dossier temporaire contenant uniquement `tokenizer.json` et `metadata.yml` ;
2. appelle `competition.evaluation.evaluate_submission()` sur le CSV validation officiel complet ;
3. conserve le rapport de `competition.validation` et de `competition.metrics` ;
4. appelle le helper épinglé avec les mêmes 24 000 lignes ;
5. compte les reconstructions strictes sans NFC, `strip()` ni tokens spéciaux ;
6. vérifie l’arithmétique du score et calcule la marge exacte ;
7. ne marque le candidat admissible qu’après comparaison au b4 réévalué.

Le score affiché par `competition` reste le score principal. Un désaccord entre les deux chemins est
conservé dans le rapport et empêche la recommandation automatique.

In [ ]:
VALIDATION_EXAMPLES = []
with VALIDATION_PATH.open(encoding="utf-8") as stream:
    for line in stream:
        language, text = json.loads(line)
        VALIDATION_EXAMPLES.append(Example(language, text))
VALIDATION_TEXTS = [item.text for item in VALIDATION_EXAMPLES]
if len(VALIDATION_EXAMPLES) != 24_000:
    raise RuntimeError("La validation complète de 24 000 lignes est obligatoire")
if Counter(item.language for item in VALIDATION_EXAMPLES) != dict.fromkeys(LANGUAGES, 4_000):
    raise RuntimeError("Répartition validation inattendue")

def copy_checked(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if sha256_file(destination) != sha256_file(source):
            raise RuntimeError(f"Refus de remplacer un fichier existant: {destination}")
        return
    temporary = destination.with_name(destination.name + ".partial")
    shutil.copyfile(source, temporary)
    if sha256_file(temporary) != sha256_file(source):
        raise RuntimeError("Erreur de checksum pendant la copie")
    os.replace(temporary, destination)

def prepare_evaluation_input(name, model_path):
    directory = EVALUATION_SUBMISSIONS / name
    directory.mkdir(parents=True, exist_ok=True)
    tokenizer_path = directory / "tokenizer.json"
    copy_checked(model_path, tokenizer_path)
    metadata_path = directory / "metadata.yml"
    if metadata_path.exists():
        return directory
    metadata_path.write_text(
        "team: Maick Dane Nkou research\n"
        "members:\n"
        "  - Maick Dane Nkou\n"
        "affiliation: research-only\n"
        "approach: candidate evaluation only; not a submission\n",
        encoding="utf-8",
    )
    return directory

def strict_failure_count(tokenizer, texts, batch_size=512):
    failures = 0
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encodings = tokenizer.encode_batch(batch, add_special_tokens=False)
        ids = [encoding.ids for encoding in encodings]
        decoded = tokenizer.decode_batch(ids, skip_special_tokens=False)
        skipped = tokenizer.decode_batch(ids, skip_special_tokens=True)
        failures += sum(
            original != restored or original != restored_skipped
            for original, restored, restored_skipped in zip(batch, decoded, skipped, strict=True)
        )
    return failures

def exact_headroom(fertility):
    raw_target_mean = sum(fertility[language] for language in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)
    budget = raw_target_mean * CONTEXT_FERTILITY_RATIO
    headroom = min(
        (budget - fertility[language]) / budget
        for language in ("en", "fr")
    )
    return raw_target_mean, budget, headroom

def check_score_arithmetic(result):
    fertility = result["fertility"]
    unknown_rate = result["unknown_rate"]
    penalised = result["penalised"]
    for language in LANGUAGES:
        expected = fertility[language] + 100.0 * unknown_rate[language]
        if not math.isclose(penalised[language], expected, rel_tol=1e-12, abs_tol=1e-12):
            raise RuntimeError(f"Arithmétique UNK incohérente pour {language}")
    raw_target_mean, budget, _ = exact_headroom(fertility)
    guardrail = sum(max(0.0, fertility[language] - budget) for language in ("en", "fr"))
    base = sum(penalised[language] for language in SCORED_LANGUAGES) / 4
    expected_score = base + guardrail + RECONSTRUCTION_PENALTY * (1.0 - result["reconstruction"])
    if not math.isclose(result["score"], expected_score, rel_tol=1e-12, abs_tol=1e-12):
        raise RuntimeError("Arithmétique du score competition incohérente")
    if not math.isclose(result["guardrail_penalty"], guardrail, rel_tol=1e-12, abs_tol=1e-12):
        raise RuntimeError("Arithmétique de la pénalité EN/FR incohérente")
    return {
        "raw_target_mean": raw_target_mean,
        "guardrail_budget": budget,
        "headroom": exact_headroom(fertility)[2],
        "base_score": base,
        "reconstruction_penalty": RECONSTRUCTION_PENALTY * (1.0 - result["reconstruction"]),
    }

def evaluate_candidate(name, model_path, config):
    model_path = Path(model_path)
    model_sha = sha256_file(model_path)
    submission_dir = prepare_evaluation_input(name, model_path)
    competition_result = evaluate_submission(
        submission_dir,
        VALIDATION_CSV,
        benchmark_repeats=3,
    )
    validation_report = validate_tokenizer(model_path)
    arithmetic = check_score_arithmetic(competition_result)
    tokenizer = tokenizers.Tokenizer.from_file(str(model_path))
    strict_failures = strict_failure_count(tokenizer, VALIDATION_TEXTS)
    pinned_report = official_utils.profile_submission(
        model_path,
        data=VALIDATION_EXAMPLES,
        repeats=1,
        verbose=False,
    )
    score_delta = competition_result["score"] - pinned_report["score"]
    score_agreement = math.isclose(
        competition_result["score"], pinned_report["score"], rel_tol=1e-12, abs_tol=1e-12
    )
    result = {
        "name": name,
        "config": config,
        "tokenizer_sha256": model_sha,
        "competition": competition_result,
        "competition_validation": validation_report.as_dict(),
        "official_checker": pinned_report,
        "score_delta_competition_minus_checker": score_delta,
        "score_agreement": score_agreement,
        "strict_failures": strict_failures,
        "arithmetic": arithmetic,
        "training": None,
        "leaderboard": None,
    }
    if model_sha != sha256_file(model_path):
        raise RuntimeError("Tokenizer modifié pendant l’évaluation")
    if pinned_report.get("rows") != 24_000 or competition_result.get("rows") != 24_000:
        raise RuntimeError("Évaluation incomplète")
    return result


## Entraînements et reprise

Les modèles terminés sont réutilisés seulement si leur configuration, le SHA du train et le SHA du
modèle correspondent au manifeste de la session. Dans tous les cas, un modèle réutilisé est réévalué ;
un ancien rapport n’est jamais considéré comme une mesure actuelle.

Une interruption d’un entraînement repart du début de cet entraînement. Elle ne reprend pas l’algorithme
BPE à l’intérieur d’un modèle.

In [ ]:
def run_process(arguments, log_path):
    environment = os.environ.copy()
    environment.update({
        "RAYON_NUM_THREADS": str(CPU_THREADS),
        "TOKENIZERS_PARALLELISM": "true",
        "PYTHONUNBUFFERED": "1",
    })
    command = [sys.executable, "-u", str(WORKER_PATH), *map(str, arguments)]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            env=environment,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
        if return_code != 0:
            raise RuntimeError(f"Travailleur terminé avec le code {return_code}; voir {log_path}")

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, value):
    atomic_write_text(path, json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n")

def complete_training(model_dir, config):
    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = model_dir / "manifest.json"
    model_path = model_dir / "tokenizer.json"
    receipt_path = model_dir / "training.json"
    expected = {"config": config, "train_sha256": TRAIN_INFO["sha256"], "worker_sha256": WORKER_SHA256}
    if manifest_path.exists():
        manifest = read_json(manifest_path)
        if manifest != expected:
            raise RuntimeError(f"Manifeste incompatible pour {config['name']}")
        receipt = read_json(receipt_path)
        if receipt["tokenizer_sha256"] != sha256_file(model_path):
            raise RuntimeError(f"SHA du modèle sauvegardé incohérent pour {config['name']}")
        print("Reprise sans entraînement :", config["name"])
        return model_path, receipt
    if any(path.exists() for path in (model_path, receipt_path)):
        raise RuntimeError(f"Sortie partielle détectée pour {config['name']}; vérifier avant reprise")
    stage = WORK_DIR / "staging" / config["name"]
    stage.mkdir(parents=True, exist_ok=True)
    stage_config = stage / "config.json"
    stage_model = stage / "tokenizer.json"
    write_json(stage_config, config)
    log_path = SESSION_DIR / "logs" / f"{config['name']}.training.log"
    print("\\n=== Entraînement :", config["name"], "===")
    run_process([TRAIN_PATH, stage_config, stage_model], log_path)
    receipt = read_json(stage / "training.json")
    if receipt["config"] != config or receipt["train_sha256"] != TRAIN_INFO["sha256"]:
        raise RuntimeError(f"Provenance d’entraînement incohérente pour {config['name']}")
    copy_checked(stage_model, model_path)
    write_json(receipt_path, receipt)
    write_json(manifest_path, expected)
    return model_path, receipt

# La référence b4 est évaluée sans entraînement.
reference_config = {
    "name": "b4-reference",
    "units": {language: 8 if language in ("en", "fr") else 32 for language in LANGUAGES},
    "denominator": DENOMINATOR,
    "vocab_size": 10_000,
    "min_frequency": 5,
    "architecture": "space_word_v1",
    "source": REFERENCE_COMMIT,
}
reference_result = evaluate_candidate("b4-reference", REFERENCE_PATH, reference_config)
reference_result["reference_sha_expected"] = REFERENCE_SHA256
reference_result["reference_sha_matches"] = reference_result["tokenizer_sha256"] == REFERENCE_SHA256
if not reference_result["reference_sha_matches"]:
    raise RuntimeError("La référence b4 ne correspond pas au SHA historique")
if reference_result["strict_failures"] != 0:
    raise RuntimeError("La référence b4 n’est pas strictement réversible sur validation")
write_json(SESSION_DIR / "reference" / "evaluation.json", reference_result)
print(
    "B4 réévalué — score competition:", f"{reference_result['competition']['score']:.12f}",
    "marge:", f"{100 * reference_result['arithmetic']['headroom']:.12f}%",
)


In [ ]:
candidate_results = [reference_result]
for candidate_config in CONFIGS:
    model_dir = SESSION_DIR / "models" / candidate_config["name"]
    model_path, receipt = complete_training(model_dir, candidate_config)
    result = evaluate_candidate(candidate_config["name"], model_path, candidate_config)
    result["training"] = receipt
    write_json(SESSION_DIR / "reports" / f"{candidate_config['name']}.json", result)
    candidate_results.append(result)
    print(
        candidate_config["name"],
        "score:", f"{result['competition']['score']:.12f}",
        "marge:", f"{100 * result['arithmetic']['headroom']:.12f}%",
        "strict_failures:", result["strict_failures"],
    )
print("Recherche terminée :", len(candidate_results), "modèles évalués, dont la référence b4.")


## Contrôle du chemin leaderboard

`competition.leaderboard.build_leaderboard()` ajoute la mesure de débit et la limite de temps relative à
la baseline lorsqu’un dossier `baseline` est présent. Pour ne pas modifier le dépôt, le notebook crée
un dossier d’entrée temporaire dans la session et y copie la baseline character-level fournie par le
répertoire de l’évaluateur.

Ce contrôle ne remplace pas les rapports complets ci-dessus. Il sert à signaler un éventuel rejet par la
règle de temps ou un départage par débit. Aucun score n’est diminué ou corrigé pour appliquer une politique
locale.

In [ ]:
# Le leaderboard officiel reconnait le dossier nommé `baseline` et applique la limite de temps
# aux dossiers parcourus après lui. Les candidats sont préfixés par `z-` pour garantir cet ordre.
LEADERBOARD_SUBMISSIONS = SESSION_DIR / "leaderboard_inputs"
LEADERBOARD_SUBMISSIONS.mkdir(parents=True, exist_ok=True)

def prepare_leaderboard_input(slug, source):
    directory = LEADERBOARD_SUBMISSIONS / slug
    directory.mkdir(parents=True, exist_ok=True)
    copy_checked(source, directory / "tokenizer.json")
    metadata_path = directory / "metadata.yml"
    if not metadata_path.exists():
        metadata_path.write_text(
            "team: Maick Dane Nkou research\n"
            "members:\n"
            "  - Maick Dane Nkou\n"
            "approach: leaderboard-only research evaluation\n",
            encoding="utf-8",
        )
    return directory

BASELINE_SOURCE = REPO_ROOT / "starter" / "baselines" / "character-level" / "tokenizer.json"
if BASELINE_SOURCE.is_file():
    prepare_leaderboard_input("baseline", BASELINE_SOURCE)
else:
    print("Baseline character-level absente : le contrôle de temps leaderboard sera indisponible.")

prepare_leaderboard_input("z-b4-reference", REFERENCE_PATH)
for candidate_config in CONFIGS:
    prepare_leaderboard_input(
        "z-" + candidate_config["name"],
        SESSION_DIR / "models" / candidate_config["name"] / "tokenizer.json",
    )

leaderboard_rows, leaderboard_failures = build_leaderboard(
    LEADERBOARD_SUBMISSIONS,
    VALIDATION_CSV,
    benchmark_repeats=3,
)
write_json(SESSION_DIR / "leaderboard_rows.json", leaderboard_rows)
write_json(SESSION_DIR / "leaderboard_failures.json", leaderboard_failures)
leaderboard_by_slug = {row["slug"]: row for row in leaderboard_rows}
leaderboard_failure_by_slug = {item["slug"]: item["error"] for item in leaderboard_failures}
for result in candidate_results:
    slug = "z-" + result["name"]
    result["leaderboard"] = {
        "row": leaderboard_by_slug.get(slug),
        "failure": leaderboard_failure_by_slug.get(slug),
    }
    report_path = SESSION_DIR / "reference" / "evaluation.json" if slug == "z-b4-reference" else SESSION_DIR / "reports" / f"{result['name']}.json"
    write_json(report_path, result)
print("Lignes leaderboard :", len(leaderboard_rows))
print("Échecs leaderboard :", leaderboard_failures)


## Sélection locale explicite

La sélection ci-dessous est une politique de recherche, pas une modification du score officiel.
Elle exige une dominance stricte sur le b4 **réévalué dans cette session**.

In [ ]:
def selection_reasons(candidate, reference):
    reasons = []
    competition_result = candidate["competition"]
    checker_result = candidate["official_checker"]
    if not candidate["score_agreement"]:
        reasons.append("désaccord competition/checker")
    if candidate["strict_failures"] != 0:
        reasons.append("reconstruction stricte non exacte")
    if checker_result.get("lossy_rows") != 0:
        reasons.append("pénalité de reconstruction checker")
    if any(value != 0.0 for value in competition_result["unknown_rate"].values()):
        reasons.append("UNK non nul")
    if competition_result["guardrail_penalty"] != 0.0:
        reasons.append("pénalité EN/FR")
    if competition_result["reconstruction"] != 1.0:
        reasons.append("reconstruction competition incomplète")
    if not candidate["score_agreement"]:
        reasons.append("scores des deux chemins différents")
    reference_headroom = reference["arithmetic"]["headroom"]
    candidate_headroom = candidate["arithmetic"]["headroom"]
    if candidate_headroom < reference_headroom - 1e-12:
        reasons.append("marge EN/FR inférieure au b4")
    if competition_result["score"] >= reference["competition"]["score"] - 1e-12:
        reasons.append("score non strictement inférieur au b4")
    if candidate["leaderboard"].get("failure"):
        reasons.append("rejet par le chemin leaderboard")
    return reasons

reference_result["selection_reasons"] = ["référence b4"]
reference_result["admissible"] = True
for result in candidate_results[1:]:
    result["selection_reasons"] = selection_reasons(result, reference_result)
    result["admissible"] = not result["selection_reasons"]
    report_path = SESSION_DIR / "reports" / f"{result['name']}.json"
    write_json(report_path, result)

admissible = [result for result in candidate_results[1:] if result["admissible"]]
best = min(admissible, key=lambda item: item["competition"]["score"]) if admissible else reference_result
print("\\nRéférence score exact :", f"{reference_result['competition']['score']:.15f}")
print("Référence marge exacte :", f"{100 * reference_result['arithmetic']['headroom']:.15f}%")
for result in candidate_results:
    print(
        result["name"],
        "score=", f"{result['competition']['score']:.15f}",
        "marge=", f"{100 * result['arithmetic']['headroom']:.15f}%",
        "admissible=", result["admissible"],
        "raison=", "; ".join(result.get("selection_reasons", [])),
    )
if best["name"] == "b4-reference":
    print("Aucune amélioration admissible démontrée : b4 reste la référence.")
else:
    print("Candidat à examiner manuellement :", best["name"])
    print("Ce notebook ne le copie pas dans submissions/.")


In [ ]:
# Tableau CSV complet : nombres écrits sans arrondi de sélection.
comparison_path = SESSION_DIR / "comparison.csv"
columns = [
    "candidate", "status", "score_competition", "score_checker", "score_delta",
    "base_score", "guardrail_penalty", "reconstruction_penalty", "headroom_percent",
    "strict_failures", "zero_unk", "score_agreement", "leaderboard_failure",
    "throughput_chars_per_second", "elapsed_seconds", "vocab_size", "sha256",
    "admissible", "reason",
]
with comparison_path.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    for result in candidate_results:
        comp = result["competition"]
        arithmetic = result["arithmetic"]
        writer.writerow({
            "candidate": result["name"],
            "status": "reference" if result["name"] == "b4-reference" else "measured",
            "score_competition": format(comp["score"], ".17g"),
            "score_checker": format(result["official_checker"]["score"], ".17g"),
            "score_delta": format(result["score_delta_competition_minus_checker"], ".17g"),
            "base_score": format(arithmetic["base_score"], ".17g"),
            "guardrail_penalty": format(comp["guardrail_penalty"], ".17g"),
            "reconstruction_penalty": format(arithmetic["reconstruction_penalty"], ".17g"),
            "headroom_percent": format(100 * arithmetic["headroom"], ".17g"),
            "strict_failures": result["strict_failures"],
            "zero_unk": all(value == 0.0 for value in comp["unknown_rate"].values()),
            "score_agreement": result["score_agreement"],
            "leaderboard_failure": result["leaderboard"].get("failure") or "",
            "throughput_chars_per_second": format(comp["throughput"], ".17g"),
            "elapsed_seconds": format(comp["elapsed_seconds"], ".17g"),
            "vocab_size": comp["vocab_size"],
            "sha256": result["tokenizer_sha256"],
            "admissible": result["admissible"],
            "reason": "; ".join(result.get("selection_reasons", [])),
        })
print("Rapport CSV :", comparison_path)


## Résultats et artefacts

À partager pour analyse :

- `comparison.csv` ;
- `run_context.json` ;
- les rapports JSON dans `reference/` et `reports/` ;
- `leaderboard_rows.json` et `leaderboard_failures.json` ;
- les SHA des modèles et les journaux d’entraînement.

Le meilleur candidat éventuel est uniquement marqué comme **candidat à examiner**. Aucun téléchargement,
aucune copie vers `submissions/` et aucune modification du tokenizer soumis ne sont automatiques.

Une petite archive de rapports peut être créée pour téléchargement Colab. Elle ne contient pas les corpus
ni les modèles, et elle n’est pas un artefact obligatoire de soumission.

In [ ]:
DOWNLOAD_REPORTS = False
if DOWNLOAD_REPORTS:
    archive_path = SESSION_DIR / "review_reports.zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in [
            SESSION_DIR / "run_context.json",
            comparison_path,
            SESSION_DIR / "leaderboard_rows.json",
            SESSION_DIR / "leaderboard_failures.json",
            SESSION_DIR / "reference" / "evaluation.json",
        ]:
            if path.is_file():
                archive.write(path, path.relative_to(SESSION_DIR))
        for path in (SESSION_DIR / "reports").glob("*.json"):
            archive.write(path, path.relative_to(SESSION_DIR))
    print("Archive de rapports :", archive_path)
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        print("Hors Colab : récupérer l’archive au chemin affiché.")
else:
    print("DOWNLOAD_REPORTS=False : aucun téléchargement automatique.")
print("Modèle de référence soumis inchangé :", REFERENCE_SHA256)
print("Le résultat est une analyse publique, pas une promesse de score caché.")
